In [4]:
# 1. Install Libraries
!pip install transformers datasets scikit-learn accelerate -U -q
import os
os.environ["WANDB_DISABLED"] = "true"
import torch
from datasets import load_dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print("✅ Libraries Installed")

# 2. Load Data (Financial PhraseBank mirror)
print("⬇️ Downloading Public Dataset (Financial PhraseBank)...")
raw_dataset = load_dataset("descartes100/enhanced-financial-phrasebank")

# This mirror wraps every row under a 'train' key — unnest it
dataset = raw_dataset["train"].map(lambda x: x["train"])
dataset = dataset.remove_columns(["train"])

print(dataset)
print(dataset[0])

# Now split into train/test
split_dataset = dataset.train_test_split(test_size=0.2, seed=42)

# 3. Tokenize (Prepare for BERT)
print("🔄 Tokenizing Data...")
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize_function(examples):
    return tokenizer(examples['sentence'], padding="max_length", truncation=True)

tokenized_datasets = split_dataset.map(tokenize_function, batched=True)

train_dataset = tokenized_datasets['train']
test_dataset = tokenized_datasets['test']

# 5. Load the Pre-Trained Model
# 3 labels: negative, neutral, positive
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=3
)

# 6. Define Training Arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="no"
)

# 7. Metrics Function
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# 8. TRAIN!
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("🚀 Starting Training... (This takes about 5-8 mins on GPU)")
trainer.train()

# 9. Save the Model
print("💾 Saving Model...")
save_path = "./my_lead_model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

# Zip it for download
!zip -r my_lead_model.zip ./my_lead_model
print("✅ DONE! Download 'my_lead_model.zip' from the files tab on the left.")

✅ Libraries Installed
⬇️ Downloading Public Dataset (Financial PhraseBank)...


Map:   0%|          | 0/4846 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'sentence'],
    num_rows: 4846
})
{'label': 1, 'sentence': "Amidst the company's rapid growth in Russia, Gran confirmed that there are no immediate plans to shift all production to the region. Despite its promising prospects in Russia, the company seems to be cautious about relocating production entirely."}
🔄 Tokenizing Data...


Map:   0%|          | 0/3876 [00:00<?, ? examples/s]

Map:   0%|          | 0/970 [00:00<?, ? examples/s]

...

🚀 Starting Training... (This takes about 5-8 mins on GPU)


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.599669,0.629155,0.691753,0.691016,0.772832,0.691753
2,0.434498,0.515654,0.788660,0.791496,0.808505,0.788660


💾 Saving Model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: my_lead_model/ (stored 0%)
  adding: my_lead_model/tokenizer_config.json (deflated 43%)
  adding: my_lead_model/model.safetensors (deflated 8%)
  adding: my_lead_model/tokenizer.json (deflated 71%)
  adding: my_lead_model/config.json (deflated 52%)
✅ DONE! Download 'my_lead_model.zip' from the files tab on the left.


In [5]:
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import os

MODEL_PATH = "./my_lead_model"

def load_local_model():
    print(f"📂 Loading model from {MODEL_PATH}...")
    try:
        tokenizer = DistilBertTokenizer.from_pretrained(MODEL_PATH)
        model = DistilBertForSequenceClassification.from_pretrained(MODEL_PATH)
        model.eval()
        print("✅ Model loaded successfully!")
        return tokenizer, model
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        return None, None

def predict_lead(text, tokenizer, model):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)

    negative_score = probs[0][0].item()
    neutral_score = probs[0][1].item()
    positive_score = probs[0][2].item()

    lead_score = positive_score + (0.5 * neutral_score)

    if lead_score >= 0.65:
        label = "High"
    elif lead_score >= 0.35:
        label = "Medium"
    else:
        label = "Low"

    return {"negative": negative_score, "neutral": neutral_score, "positive": positive_score, "lead_score": lead_score}, label

if __name__ == "__main__":
    tokenizer, model = load_local_model()

    if model:
        print("\n🧪 TESTING ON SALES NOTES (Proxy Method)...")
        print("-" * 70)
        print(f"{'INPUT TEXT':<50} | {'SCORE':<6} | {'LABEL'}")
        print("-" * 70)

        test_cases = [
            "Client absolutely loved the demo and wants a contract immediately.",
            "Budget is approved, they are excited to move forward.",
            "Very impressed with the features, huge upgrade over their current tool.",
            "They are interested but need to check with the CEO first.",
            "The demo went okay. They have some questions about pricing.",
            "They hated the pricing model. Too expensive.",
            "Not interested at all. They are happy with their current vendor.",
            "Complaint: The software is confusing and slow. Waste of time."
        ]

        for text in test_cases:
            scores, label = predict_lead(text, tokenizer, model)
            color = "\033[92m" if label == "High" else "\033[93m" if label == "Medium" else "\033[91m"
            reset = "\033[0m"
            print(f"{text[:47] + '...':<50} | {scores['lead_score']:.4f} | {color}{label}{reset}")

📂 Loading model from ./my_lead_model...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ Model loaded successfully!

🧪 TESTING ON SALES NOTES (Proxy Method)...
----------------------------------------------------------------------
INPUT TEXT                                         | SCORE  | LABEL
----------------------------------------------------------------------
Client absolutely loved the demo and wants a co... | 0.7163 | High
Budget is approved, they are excited to move fo... | 0.5398 | Medium
Very impressed with the features, huge upgrade ... | 0.9596 | High
They are interested but need to check with the ... | 0.5003 | Medium
The demo went okay. They have some questions ab... | 0.4217 | Medium
They hated the pricing model. Too expensive....    | 0.2366 | Low
Not interested at all. They are happy with thei... | 0.7415 | High
Complaint: The software is confusing and slow. ... | 0.2029 | Low


In [6]:
print(dataset.features["label"])

Value('int64')


In [7]:
csv_text = """sentence,label
"Not interested in moving forward at this time",0
"Decided against our solution after the demo",0
"Went with our competitor instead",0
"Said they're happy with their current provider",0
"Budget is completely frozen for the rest of the year",0
"Too expensive for their current needs",0
"Renewed with the incumbent vendor",0
"Price is way over their budget",0
"Not a priority for them right now",0
"Chose the cheaper alternative from a competitor",0
"Staying with existing vendor, no interest in switching",0
"Can't justify the ROI at this price point",0
"Ghosted after the last follow-up email",0
"Said no to the proposal outright",0
"Competitor offered a better feature set for less",0
"Budget cut, so they're not buying anything",0
"Not a good fit for their use case",0
"Already signed with another provider",0
"Complained about the setup fee being too high",0
"Declined to schedule a second meeting",0
"Lost the deal to a rival vendor",0
"Said our platform lacks the integrations they need",0
"Price objection was too strong to overcome",0
"They're going with a free alternative",0
"No budget allocated for new software this quarter",0
"Unresponsive to calls and emails for weeks",0
"Decided to stick with their legacy system",0
"Too many hidden costs in our pricing",0
"Competitor gave them a steep discount",0
"Not interested in any sales pitches right now",0
"Said we're too slow to respond to support issues",0
"Passed on the opportunity due to internal changes",0
"Renewed with their current vendor for three years",0
"Budget is tied up in other projects",0
"Found a better solution elsewhere",0
"Said our product is overpriced compared to others",0
"No longer engaging with our sales team",0
"Went dark after the pricing was shared",0
"Decided against upgrading to the premium tier",0
"Competitor's onboarding process is easier, they said",0
"Budget freeze means no new contracts until next fiscal",0
"Told us they're not in the market right now",0
"Disappointed with the trial experience",0
"Chose to build an in-house solution instead",0
"Said our monthly fees are too steep for their size",0
"Lost interest after the discovery call",0
"Vendor lock-in with current provider is too strong",0
"No funds to allocate for our tool",0
"Rejected the final proposal without counteroffers",0
"Said they're evaluating but never replied again",0
"Complained that our implementation takes too long",0
"Went with a competitor that has better analytics",0
"Budget was slashed, so they can't proceed",0
"Not willing to switch from their current setup",0
"Said our support is not responsive enough",0
"Declined the meeting invitation for next week",0
"Too many bugs in the beta version, they said",0
"Renewed with the old vendor despite our discount",0
"Price is a non-starter for their finance team",0
"Said they'll stick with what they have",0
"No decision maker available, but they're not buying",0
"Competitor undercut our pricing by 30%",0
"Disliked the user interface during the demo",0
"Budget is zero for new initiatives this year",0
"Decided to postpone indefinitely",0
"Said our contract terms are too restrictive",0
"Ghosted after the trial expired",0
"Not impressed by our case studies",0
"Chose a cheaper point solution over our platform",0
"Said they're happy with the status quo",0
"No budget and no appetite for change",0
"Said our product is missing a critical feature",0
"Went with a competitor who offered free migration",0
"Told us to stop calling them",0
"Price increase made them reconsider and decline",0
"Said they've already made a decision elsewhere",0
"Not a good time, and they're not interested later either",0
"Competitor's solution is more mature, they think",0
"Said our onboarding is too complex",0
"Decided against because of negative reviews they saw",0
"Need to think about it and get back to us",1
"Checking with their team before making a decision",1
"Budget review is scheduled for next month",1
"Maybe we can revisit this in Q4",1
"Still comparing us to two other vendors",1
"Some questions remain about the implementation",1
"Waiting for the procurement team to approve",1
"Not right now, but maybe in the new year",1
"They're in the evaluation phase currently",1
"Need to run this by their IT department first",1
"Considering both our platform and a competitor's",1
"Sent them the pricing sheet, waiting for feedback",1
"Left a voicemail, waiting for a callback",1
"No final decision yet, still gathering info",1
"They have some internal discussions pending",1
"Budget is being finalized for next quarter",1
"Not sure if they can get approval this cycle",1
"Will get back to us after their weekly meeting",1
"Interested but want to see a live demo first",1
"Need to align with their partners on this",1
"Waiting to see if the budget gets released",1
"They're asking for more case studies",1
"Currently in the due diligence stage",1
"Maybe Q3, they said, but not confirmed",1
"Checking references from current customers",1
"Need to compare our SLAs with others",1
"Sent a calendar invite for a follow-up chat",1
"No urgency, but they're open to learning",1
"They're weighing the pros and cons",1
"Waiting for a sign-off from the VP",1
"Asked for an extended trial period",1
"Not ready to commit, but keeping us in mind",1
"Need to understand the security compliance better",1
"Budget is there but not allocated yet",1
"They're in the early research stage",1
"Will decide after they see the ROI numbers",1
"Checking if we integrate with their CRM",1
"Maybe next month, depending on cash flow",1
"Sent the contract, waiting for their review",1
"They have a few more demos scheduled with others",1
"Need to get feedback from the end-users",1
"Not a priority this week, but maybe later",1
"Still negotiating internally on requirements",1
"Looking at our pricing versus features",1
"They're taking a slow approach to this",1
"Will update us after their board meeting",1
"Considering a pilot program first",1
"Need to check the data migration process",1
"They're not rejecting, but not advancing yet",1
"Waiting for a competitor's final quote",1
"Budget approval is in progress, not finalized",1
"Asked for more details on the support plan",1
"Maybe in the next fiscal year",1
"They're in a holding pattern right now",1
"Need to review the legal terms carefully",1
"Checking if they can get a discount for multi-year",1
"Sent a reminder, awaiting a reply",1
"They have some concerns but are still talking",1
"No decision timeline given yet",1
"Evaluating our solution alongside others",1
"Need to see a full feature list before deciding",1
"Budget is under review by finance",1
"They're interested but cautious",1
"Will get back to us after the holidays",1
"Still in the information gathering phase",1
"Asked about implementation timelines",1
"Not saying yes, but not saying no either",1
"Need to discuss with their remote teams",1
"Maybe they'll move forward if we offer training",1
"Waiting for the technical assessment results",1
"They're comparing our uptime guarantee",1
"Budget allocated but needs final approval",1
"Wants to see a proof of concept first",1
"No rush, they're just exploring options",1
"Need to align with their strategic goals",1
"Checking if our product complies with their policies",1
"Will let us know after they meet with stakeholders",1
"They're in the negotiation phase",1
"Asked for a reference call with a customer",1
"Still thinking about the total cost of ownership",1
"Ready to sign the contract today",2
"Loved the demo and wants to move forward",2
"Approved the budget for our solution",2
"Very excited about the product's capabilities",2
"Wants to start implementation as soon as possible",2
"Chose us over the competitor",2
"Said they're ready to buy",2
"Great meeting, they're fully onboard",2
"Sent the signed agreement back",2
"They prefer our platform over the others",2
"Budget cleared, let's proceed with the proposal",2
"Eager to get their team trained on our tool",2
"Confirmed they're switching from the old vendor",2
"Loved the ROI numbers we presented",2
"Wants to move to the next stage of the deal",2
"Finance gave the green light for this purchase",2
"They're impressed with our customer support",2
"Ready to schedule the kickoff call",2
"Said our solution is exactly what they needed",2
"Asked for the invoice so they can pay",2
"They're recommending us to their sister company",2
"Excited about the analytics features we showed",2
"Wants to sign a multi-year agreement",2
"Said we beat the competitor's pricing and features",2
"Ready to onboard next week",2
"Very positive feedback from their entire team",2
"They're convinced and want to finalize",2
"Approved by the procurement committee",2
"Can't wait to get started with our platform",2
"Said the trial exceeded their expectations",2
"Wants to add more users to the pilot",2
"They're confident in our ability to deliver",2
"Sent the deposit payment",2
"Chose our premium plan over the basic",2
"Highly enthusiastic about the roadmap",2
"Ready to replace their existing system with ours",2
"Said they're 100% on board",2
"Wants to fast-track the implementation",2
"Loved the customization options we offered",2
"They're referring us to three other businesses",2
"Budget is approved and ready to spend",2
"Said our demo was the best they've seen",2
"Wants to close the deal before the quarter ends",2
"They're excited about the integration capabilities",2
"Ready to issue the purchase order",2
"Said we won them over against the incumbent",2
"Very satisfied with the pricing negotiation",2
"They're preparing the internal announcement",2
"Wants to sign up for the annual subscription",2
"Loved the white-glove onboarding idea",2
"Said our solution is a game-changer for them",2
"Ready to deploy the software across their teams",2
"They're thrilled with the support response time",2
"Approved the SOW and wants to execute",2
"Said they're glad they found us",2
"Wants to increase the order quantity",2
"They're convinced about the long-term value",2
"Ready to proceed without any further hesitation",2
"Loved the case studies we shared",2
"Said our pricing is fair for the value",2
"Wants to schedule the implementation kickoff",2
"They're eager to leave their current provider",2
"Positive sign: asked for a direct line to support",2
"Ready to make a decision today",2
"Said we're their top choice",2
"Approved the budget variance for our tool",2
"Wants to roll out to all their locations",2
"Excited about the AI features in our product",2
"They're pushing for a quick start date",2
"Said our competitor's offering doesn't compare",2
"Very receptive to the upsell options",2
"Ready to finalize the legal paperwork",2
"They've already told their staff about the switch",2
"Loved the dashboard and reporting",2
"Said they're tired of their old vendor and want us",2
"Wants to get the contract countersigned today",2
"They're very optimistic about the partnership",2
"Approved the pilot expansion to full deployment",2
"Said we addressed all their pain points perfectly",2
"Ready to commit to a 5-year deal",2
"""

with open("sales_leads_finetune.csv", "w") as f:
    f.write(csv_text)

import pandas as pd
df = pd.read_csv("sales_leads_finetune.csv")
print(df.shape)
print(df["label"].value_counts())
print(df.head())

(240, 2)
label
0    80
1    80
2    80
Name: count, dtype: int64
                                            sentence  label
0      Not interested in moving forward at this time      0
1        Decided against our solution after the demo      0
2                   Went with our competitor instead      0
3     Said they're happy with their current provider      0
4  Budget is completely frozen for the rest of th...      0


In [8]:
# 1. Install (if fresh session)
!pip install transformers datasets scikit-learn accelerate -U -q
import os
os.environ["WANDB_DISABLED"] = "true"
import torch
import pandas as pd
from datasets import Dataset
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print("✅ Libraries Installed")

# 2. Load your rule-based CSV
df = pd.read_csv("sales_leads_finetune.csv")
print(df["label"].value_counts())  # sanity check: should be 80/80/80

full_dataset = Dataset.from_pandas(df)
split_dataset = full_dataset.train_test_split(test_size=0.2, seed=42)

# 3. Load tokenizer + your ALREADY fine-tuned model (not base distilbert)
MODEL_PATH = "./my_lead_model"
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_PATH)
model = DistilBertForSequenceClassification.from_pretrained(MODEL_PATH)

def tokenize_function(examples):
    return tokenizer(examples['sentence'], padding="max_length", truncation=True)

tokenized_datasets = split_dataset.map(tokenize_function, batched=True)
train_dataset = tokenized_datasets['train']
test_dataset = tokenized_datasets['test']

# 4. Training Arguments — LOW learning rate so it adapts without forgetting the base
training_args = TrainingArguments(
    output_dir='./results_stage2',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,          # deliberately low — this is a refinement, not a fresh train
    warmup_steps=20,
    weight_decay=0.01,
    logging_dir='./logs_stage2',
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="no"
)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

# 5. TRAIN (stage 2 — domain adaptation)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("🚀 Starting Stage 2 Fine-Tuning (Sales Domain Adaptation)...")
trainer.train()

# 6. Save the updated model
print("💾 Saving Domain-Adapted Model...")
save_path = "./my_lead_model_v2"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

!zip -r my_lead_model_v2.zip ./my_lead_model_v2
print("✅ DONE! Download 'my_lead_model_v2.zip' from the files tab on the left.")

✅ Libraries Installed
label
0    80
1    80
2    80
Name: count, dtype: int64


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Map:   0%|          | 0/192 [00:00<?, ? examples/s]

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


🚀 Starting Stage 2 Fine-Tuning (Sales Domain Adaptation)...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.794801,0.743218,0.666667,0.670833,0.708333,0.666667
2,0.507858,0.506350,0.854167,0.852747,0.852572,0.854167
3,0.215996,0.483048,0.854167,0.854299,0.857487,0.854167


💾 Saving Domain-Adapted Model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: my_lead_model_v2/ (stored 0%)
  adding: my_lead_model_v2/tokenizer_config.json (deflated 50%)
  adding: my_lead_model_v2/model.safetensors (deflated 8%)
  adding: my_lead_model_v2/tokenizer.json (deflated 71%)
  adding: my_lead_model_v2/config.json (deflated 52%)
✅ DONE! Download 'my_lead_model_v2.zip' from the files tab on the left.


In [9]:
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import os

MODEL_PATH = "./my_lead_model_v2"

def load_local_model():
    print(f"📂 Loading model from {MODEL_PATH}...")
    try:
        tokenizer = DistilBertTokenizer.from_pretrained(MODEL_PATH)
        model = DistilBertForSequenceClassification.from_pretrained(MODEL_PATH)
        model.eval()
        print("✅ Model loaded successfully!")
        return tokenizer, model
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        return None, None

def predict_lead(text, tokenizer, model):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)

    negative_score = probs[0][0].item()
    neutral_score = probs[0][1].item()
    positive_score = probs[0][2].item()

    lead_score = positive_score + (0.5 * neutral_score)

    if lead_score >= 0.65:
        label = "High"
    elif lead_score >= 0.35:
        label = "Medium"
    else:
        label = "Low"

    return {"negative": negative_score, "neutral": neutral_score, "positive": positive_score, "lead_score": lead_score}, label

if __name__ == "__main__":
    tokenizer, model = load_local_model()

    if model:
        print("\n🧪 TESTING ON FRESH SALES NOTES (v2 - Domain Adapted)...")
        print("-" * 70)
        print(f"{'INPUT TEXT':<50} | {'SCORE':<6} | {'LABEL'}")
        print("-" * 70)

        test_cases = [
            # Repeat of the original failure case — must flip to Low
            "Not interested at all. They are happy with their current vendor.",

            # New negation/competitor cases (not in training set — true generalization test)
            "They said no thanks, sticking with what they already use.",
            "Client is satisfied with their existing tool and doesn't want to switch.",
            "Passed on us, going with a rival company instead.",
            "Told us they're not looking to change providers this year.",

            # New clear positive cases
            "They want to sign this week and start onboarding immediately.",
            "Really impressed, asked for the contract right after the call.",

            # New neutral/hedging cases
            "Still weighing a few options, nothing decided yet.",
            "Said they'd loop in their manager before responding.",

            # Tricky mixed-signal case
            "Liked the product but budget got cut this quarter.",
            "Interested in principle, but their current contract doesn't expire for months."
        ]

        for text in test_cases:
            scores, label = predict_lead(text, tokenizer, model)
            color = "\033[92m" if label == "High" else "\033[93m" if label == "Medium" else "\033[91m"
            reset = "\033[0m"
            print(f"{text[:47] + '...':<50} | {scores['lead_score']:.4f} | {color}{label}{reset}")

📂 Loading model from ./my_lead_model_v2...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ Model loaded successfully!

🧪 TESTING ON FRESH SALES NOTES (v2 - Domain Adapted)...
----------------------------------------------------------------------
INPUT TEXT                                         | SCORE  | LABEL
----------------------------------------------------------------------
Not interested at all. They are happy with thei... | 0.8805 | High
They said no thanks, sticking with what they al... | 0.1755 | Low
Client is satisfied with their existing tool an... | 0.3380 | Low
Passed on us, going with a rival company instea... | 0.0849 | Low
Told us they're not looking to change providers... | 0.0553 | Low
They want to sign this week and start onboardin... | 0.9141 | High
Really impressed, asked for the contract right ... | 0.9750 | High
Still weighing a few options, nothing decided y... | 0.3235 | Low
Said they'd loop in their manager before respon... | 0.4277 | Medium
Liked the product but budget got cut this quart... | 0.0307 | Low
Interested in principle, but their cur

In [10]:
# 1. Zip the folder (You cannot download folders directly)
!zip -r my_lead_model_v2.zip my_lead_model_v2

# 2. Trigger the download to your computer
from google.colab import files
files.download('my_lead_model_v2.zip')

updating: my_lead_model_v2/ (stored 0%)
updating: my_lead_model_v2/tokenizer_config.json (deflated 50%)
updating: my_lead_model_v2/model.safetensors (deflated 8%)
updating: my_lead_model_v2/tokenizer.json (deflated 71%)
updating: my_lead_model_v2/config.json (deflated 52%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>